# ETL — Adição / Correção de `source_id` e `source`

Itera sobre todos os CSVs em `INPUT_DIR`, corrige ou adiciona as colunas `source_id` e `source` de acordo com o mapeamento definido, e escreve os resultados em `OUTPUT_DIR` mantendo a estrutura de subpastas.

**Separador:** `;` | **Encoding:** `utf-8` (sem BOM) | **Index:** não escrito

---
## Célula 1 — Imports e configuração de paths

In [ ]:
import pandas as pd
import fnmatch
from pathlib import Path

# ─── PATHS ────────────────────────────────────────────────────────────────────
# Pasta raiz que contém as subpastas aima_sef/, ine/, pordata/, seg-social/, …
#INPUT_DIR  = Path("data/2-clean")
# Pasta de destino — recriará a mesma estrutura de subpastas
#OUTPUT_DIR = Path("data/3-delivery")

REPO_ROOT  = Path.cwd().parents[1]   # sobe: validacao/ → scripts/ → repo raiz
INPUT_DIR  = REPO_ROOT / "data" / "2-clean"
OUTPUT_DIR = REPO_ROOT / "data" / "3-delivery" / "seg_social" / "fontes_individuais"

SEP      = ";"
ENCODING = "utf-8"   # pandas escreve utf-8 sem BOM por defeito

print(f"Input : {INPUT_DIR.resolve()}")
print(f"Output: {OUTPUT_DIR.resolve()}")

print("Existe?", INPUT_DIR.exists())

---
## Célula 2 — Dicionário de mapeamento

Cada entrada mapeia um **padrão glob** (relativo a `INPUT_DIR`) para um tuplo `(source_id, source)`.  
O dicionário é percorrido **por ordem**, portanto padrões mais específicos devem vir primeiro.

In [ ]:
# ─── MAPEAMENTO ───────────────────────────────────────────────────────────────
# Chave  : padrão glob relativo a INPUT_DIR  (suporta **, *, ?)
# Valor  : (source_id, source)
#
# ORDEM IMPORTA: o primeiro padrão que fizer match é usado.
# Coloca padrões mais específicos (nome completo) antes dos genéricos (rifa_*).

SOURCE_MAP: dict[str, tuple[str, str]] = {

    # ── Ficheiro combinado SEF + AIMA (mais específico — vem primeiro) ─────────
    "aima_sef/residents_permits_nationality_and_gender_2015_2024.csv": (
        "F001 / F002",
        "SEF + AIMA — combinado",
    ),

    # ── SEF ───────────────────────────────────────────────────────────────────
    "aima_sef/rifa_*.csv": (
        "F001",
        "SEF - Serviço de Estrangeiros e Fronteiras",
    ),

    # ── AIMA ──────────────────────────────────────────────────────────────────
    "aima_sef/rma_*.csv": (
        "F002",
        "AIMA - Agência para a Integração, Migrações e Asilo",
    ),
    "aima_sef/series_resident_population_evolution_1980_2024.csv": (
        "F002",
        "AIMA - Agência para a Integração, Migrações e Asilo",
    ),
    "aima_sef/comparison_rma2023_rma2024.csv": (
        "F002",
        "AIMA - Agência para a Integração, Migrações e Asilo",
    ),

    # ── INE (pasta ine/) ──────────────────────────────────────────────────────
    "ine/portugal_gdp_*.csv": (
        "F007",
        "INE - Instituto Nacional de Estatística",
    ),
    "ine/portugal_gov_balance_*.csv": (
        "F007",
        "INE - Instituto Nacional de Estatística",
    ),

    # ── Pordata ───────────────────────────────────────────────────────────────
    "pordata/pordata_crude_immigration_rate_1998_2024.csv": (
        "F020",
        "Pordata",
    ),
    # Padrão genérico Pordata (F009) — fica após o F020 mais específico
    "pordata/portugal_gdp_annual_1960_2025.csv": (
        "F009",
        "Pordata",
    ),
    "pordata/pordata_foreign_resident_population_1960_2023.csv": (
        "F009",
        "Pordata",
    ),

    # ── Segurança Social / MTSSS ──────────────────────────────────────────────
    "seg_social/*": (
        "F014",
        "Segurança Social",
    ),
        #"seg_social/*.csv": (
        #"F014",
        #"Segurança Social",
    #),
    

    # ── INE ERendimentoNLocal — pendente de limpeza ───────────────────────────
    # Descomenta quando os ficheiros estiverem disponíveis em data/2-clean/
    # "ine/ERendimentoNLocal_*.csv": (
    #     "F015",
    #     "INE - Instituto Nacional de Estatística",
    # ),
}

print(f"{len(SOURCE_MAP)} padrões de mapeamento carregados.")

In [ ]:
import os
from pathlib import Path

print("Working directory:", Path.cwd())
print("INPUT_DIR absoluto:", INPUT_DIR.resolve())
print("Existe?", INPUT_DIR.exists())
print()

# Mostra o que há a partir do cwd
for p in sorted(Path.cwd().iterdir()):
    print(" ", p)

---
## Célula 3 — Função de transformação

In [ ]:
def lookup_source(relative_path: str) -> tuple[str, str] | None:
    """Devolve (source_id, source) para o caminho relativo dado, ou None se sem match."""
    rel = relative_path.replace("\\", "/")
    for pattern, mapping in SOURCE_MAP.items():
        if fnmatch.fnmatch(rel, pattern):
            return mapping
    return None


def process_csv(
    input_path: Path,
    output_path: Path,
    source_id: str,
    source: str,
) -> dict:
    """
    Lê um CSV, garante 'year' como 1ª coluna (se existir),
    adiciona/corrige source_id e source no final, e escreve no output.
    """
    df = pd.read_csv(input_path, sep=SEP, encoding=ENCODING, dtype=str)
    n_rows, _ = df.shape

    had_source_cols = "source_id" in df.columns and "source" in df.columns
    old_source_id = df["source_id"].iloc[0] if had_source_cols else None
    old_source    = df["source"].iloc[0]    if had_source_cols else None

    # 1. Atualiza/adiciona colunas de source
    df["source_id"] = source_id
    df["source"]    = source

    # 2. Reordena colunas: 'year' (se existir) -> intermédias -> 'source_id' e 'source'
    cols = list(df.columns)
    other_cols = [c for c in cols if c not in ("year", "source_id", "source")]

    if "year" in cols:
        new_order = ["year"] + other_cols + ["source_id", "source"]
    else:
        new_order = other_cols + ["source_id", "source"]

    df = df[new_order]

    # 3. Escreve o ficheiro de saída
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, sep=SEP, encoding=ENCODING, index=False)

    action = "corrigido" if had_source_cols else "adicionado"
    return {
        "ficheiro"     : str(input_path.relative_to(INPUT_DIR)),
        "ação"         : action,
        "source_id_ant": old_source_id,
        "source_id_nov": source_id,
        "source_ant"   : old_source,
        "source_nov"   : source,
        "linhas"       : n_rows,
    }


print("Funções atualizadas com sucesso.")

---
## Célula 4 — Execução e preview dos resultados

In [ ]:
results   = []   # ficheiros processados com sucesso
skipped   = []   # ficheiros sem match no mapeamento
errors    = []   # ficheiros com erro de leitura/escrita

#all_csvs = sorted(INPUT_DIR.rglob("*.csv"))
all_csvs = sorted((INPUT_DIR / "seg_social").rglob("*.csv"))
print(f"CSVs encontrados em '{INPUT_DIR}': {len(all_csvs)}\n")

for csv_path in all_csvs:
    rel = csv_path.relative_to(INPUT_DIR)
    mapping = lookup_source(str(rel))

    if mapping is None:
        skipped.append(str(rel))
        print(f"  ⚠️  SEM MATCH — {rel}")
        continue

    source_id, source = mapping
    out_path = OUTPUT_DIR / rel

    try:
        summary = process_csv(csv_path, out_path, source_id, source)
        results.append(summary)
        icon = "✏️ " if summary["ação"] == "corrigido" else "➕"
        print(f"  {icon} {summary['ação'].upper():10s} {rel}  →  source_id={source_id}")
    except Exception as exc:
        errors.append({"ficheiro": str(rel), "erro": str(exc)})
        print(f"  ❌ ERRO       {rel}  —  {exc}")

print(f"""
─────────────────────────────────────────────
Resumo
  Processados : {len(results)}
  Sem match   : {len(skipped)}
  Erros       : {len(errors)}
─────────────────────────────────────────────""")

In [ ]:
# Preview tabular dos ficheiros processados
if results:
    df_log = pd.DataFrame(results)
    display(df_log)

if skipped:
    print("\n⚠️  Ficheiros sem match no mapeamento (não foram copiados):")
    for f in skipped:
        print(f"   {f}")

if errors:
    print("\n❌ Ficheiros com erro:")
    for e in errors:
        print(f"   {e['ficheiro']}  —  {e['erro']}")

---
## Célula 5 — Escrita dos ficheiros de output

> Os ficheiros já foram escritos na Célula 4 durante a execução.  
> Esta célula serve para **validação final**: lê uma amostra de cada subpasta e confirma que as colunas estão correctas.

In [ ]:
# Validação: para cada subpasta do output, mostra o primeiro ficheiro
output_csvs = sorted(OUTPUT_DIR.rglob("*.csv"))

if not output_csvs:
    print("Nenhum ficheiro encontrado em OUTPUT_DIR — verifica se a Célula 4 correu sem erros.")
else:
    # Amostra: primeiro CSV de cada subpasta
    seen_dirs = set()
    samples   = []
    for p in output_csvs:
        if p.parent not in seen_dirs:
            seen_dirs.add(p.parent)
            samples.append(p)

    for sample in samples:
        rel = sample.relative_to(OUTPUT_DIR)
        df_check = pd.read_csv(sample, sep=SEP, encoding=ENCODING, dtype=str, nrows=3)
        print(f"\n── {rel} ──")
        # Mostra só as colunas de interesse + primeiras duas colunas de dados
        cols_show = list(df_check.columns[:2]) + [c for c in ["source_id", "source"] if c in df_check.columns]
        display(df_check[cols_show])

print(f"\n✅ Validação concluída. {len(output_csvs)} ficheiro(s) em '{OUTPUT_DIR}'.")